## Wikipedia API Testing

Search funcitonality

In [ ]:
import requests

def test_wikipedia_search(query: str, limit: int = 5):
    url = "https://en.wikipedia.org/w/rest.php/v1/search/page"
    
    # Wikimedia requires a descriptive User-Agent
    headers = {
        "User-Agent": "[REPLACE ME]"
    }
    
    params = {
        "q": query,
        "limit": limit
    }
    
    response = requests.get(url, headers=headers, params=params)

    # text block to give back to the llm instead of raw json
    llm_friendly = ""
    
    if response.status_code == 200:
        data = response.json()
        llm_friendly += f"--- Search Results for '{query}' ---\n"
        for page in data.get("pages", []):
            llm_friendly += f"Title: {page['title']}\n"
            llm_friendly += f"Description: {page.get('description', 'No description')}\n"
            llm_friendly += "-" * 20 + "\n"
    else:
        llm_friendly += f"Error: {response.status_code}\n"
        llm_friendly += response.text

    return llm_friendly

# Run the test
print(test_wikipedia_search("Quantum computing"))

Unstructured parsing logic

In [ ]:
from unstructured.partition.html import partition_html

def parse_wikipedia_html(html_content: str):
    print("Parsing HTML with unstructured...")
    
    # We pass the raw string directly to the 'text' parameter
    elements = partition_html(text=html_content)
    
    # unstructured returns a list of Element objects. 
    # Let's see how it categorized the first few pieces of the article:
    print("\n--- Element Categorization ---")
    for el in elements[:5]:
        print(f"[{type(el).__name__}]: {str(el)[:50]}...")
        
    # Now, let's filter out the junk and rebuild a clean document.
    # We only want the meaty parts: Titles, Narrative Text, and Lists.
    desired_categories = ["Title", "NarrativeText", "ListItem"]
    
    clean_paragraphs = []
    for el in elements:
        if type(el).__name__ in desired_categories:
            # We can clean up some common Wikipedia artifacts here if needed
            text = str(el).strip()
            if text:
                clean_paragraphs.append(text)
                
    # Join the clean elements with double newlines for excellent LLM readability
    final_markdown_ish_text = "\n\n".join(clean_paragraphs)
    
    return final_markdown_ish_text

Inspect Page functionality

In [ ]:
import requests

def test_wikipedia_inspect(title: str):
    formatted_title = title.replace(" ", "_")
    
    # Pointing directly to the HTML endpoint instead of /bare
    url = f"https://en.wikipedia.org/w/rest.php/v1/page/{formatted_title}/html"
    
    headers = {
        "User-Agent": "[REPLACE ME]"
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        # Because the response is raw HTML, we use .text instead of .json()
        html_content = response.text
        
        print(f"--- Raw HTML for '{title}' ---")
        
        markdown_content = parse_wikipedia_html(html_content)
        print(markdown_content)
        return markdown_content
        
    else:
        print(f"Error: {response.status_code}")
        print(response.text)

# Run the test
test_wikipedia_inspect("Quantum computing")